# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SalehAl-Nassar/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=True)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I’m picking **Lane 2 — Refresh / Content Opportunity Scoring**. The starter data tells a clear story: 54% of the 30,000 pages in this sample are on a downward trend, and 44% of all pages are both declining and drawing meaningful search demand (100+ impressions over 90 days). That’s a large pool of candidates where someone has to decide which ones to act on first. The lane guide already validates that a learned ranking beats a hand-coded rule here — Precision@50 jumps from 0.240 (baseline) to 0.740 (random forest) on this same data. That gap says the problem is real and the signals are there. I don’t need to force-fit a different lane; the numbers point at this one.

In [1]:
# (Section 1 is a text section — code lives in Section 3)

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one content item — a single pseudonymized page.

**Output:** a ranked queue of pages, each with a refresh-priority score and one or more reason codes explaining why it scored that way.

**Decision it improves:** a content reviewer deciding which pages to audit for possible refresh, expansion, or monitoring. Without a ranked queue, the reviewer faces 30,000 pages (or millions in the warehouse) with no clear starting point. With a queue, they start at the top.

**Action a human takes:** open the page, inspect it, decide whether to revise the content, update metadata, merge with another page, or leave it alone. The model doesn’t replace that judgment — it prioritises the reviewer’s time.

**Cost of a wrong recommendation:**
- *False positive (recommending a page that didn’t need attention):* the reviewer spends 20–30 minutes auditing a page that was fine. The main cost is opportunity cost — that time could have gone to a real problem.
- *False negative (missing a page that truly needs a refresh):* the page stays in decline, traffic continues dropping, and the next review window (maybe weeks or months away) is the earliest catch-up. The cost compounds over time.
- The asymmetry matters: false negatives are more expensive, so the ranking should favour recall among high-visibility decliners.

**Why ML helps instead of just eyeballing the data:** a reviewer can scan dashboards for one or two signals at a time (e.g. “show me declining pages with lots of impressions”). But the decision involves multiple signals — trend direction, impressions, position, CTR, content age, engagement, scroll rate — and their interactions shift depending on the page type and client. A learned model catches patterns that a hand-coded SQL filter would miss, and it produces a single ranked list instead of ten overlapping filter results.

This is decision-support, not proof of causation. The model says “look here first,” not “edit this page and it will recover.”

In [2]:
# (Section 2 is a text section — code lives in Section 3)

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

A note on where the numbers come from: the Precision@50 gap (baseline 0.240 vs random forest 0.740) is documented in the starter pipeline’s own verified run at `outputs/model_report.md` — I did not compute those figures here in this notebook. The percentages below (54.2%, 43.8%, 32.5%, etc.) **are** numbers I computed myself, directly from `data/raw/content_refresh_anonymized.csv`, in the code cell that follows.

In [3]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Total rows: {len(df)} unique clients: {df.client_id.nunique()}')
print()

# All 30k rows pass the standard filter (pre-filtered for us)
# 1. How many are declining, and how many of those have real demand?
n_declining = (df.trend_direction == 'down').sum()
n_declining_with_demand = ((df.trend_direction == 'down') & (df.impressions_90d >= 100)).sum()
print(f'Declining pages: {n_declining} / {len(df)} ({n_declining/len(df)*100:.1f}%)')
print(f'...of which also have 100+ impressions: {n_declining_with_demand} ({n_declining_with_demand/len(df)*100:.1f}% of all pages)')
print()

# 2. Median position and low-CTR count
median_pos = df.avg_position.median()
n_low_ctr_visible = ((df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20) & (df.ctr < 0.5)).sum()
print(f'Median avg_position: {median_pos} (50% of pages sit around page 1-2 or worse)')
print(f'Pages with 500+ impressions, pos 1-20, and low CTR (<0.5%): {n_low_ctr_visible} ({n_low_ctr_visible/len(df)*100:.1f}%)')
print()

# 3. Content age distribution
print('Content age (days):')
print(df.content_age_days.describe().to_string())
print(f'  Pages older than 6 months (>=180 days): {(df.content_age_days >= 180).sum()} / {len(df)}')
print()

# Bonus: trend direction breakdown
print('Trend direction breakdown:')
print(df.trend_direction.value_counts().to_string())
print()

# Baseline reason code hit rates
print('Baseline reason code hit rates (how many pages each rule flags):')
d = df
svp = ((d.days_since_last_update >= 180) & (d.impressions_90d >= 500)).sum()
dwd = ((d.trend_direction == 'down') & (d.impressions_90d >= 100)).sum()
pod = ((d.avg_position > 0) & (d.avg_position <= 10) & (d.content_age_days >= 180)).sum()
lcv = ((d.impressions_90d >= 500) & (d.avg_position > 0) & (d.avg_position <= 20) & (d.ctr < 0.5)).sum()
lev = ((d.sessions_90d >= 30) & ((d.engagement_rate < 30) | (d.scroll_rate < 30))).sum()
print(f'  stale_visible_page: {svp}  ({svp/len(d)*100:.1f}%)')
print(f'  declining_with_demand: {dwd}  ({dwd/len(d)*100:.1f}%)')
print(f'  page_one_decay_risk: {pod}  ({pod/len(d)*100:.1f}%)')
print(f'  low_ctr_visible_page: {lcv}  ({lcv/len(d)*100:.1f}%)')
print(f'  low_engagement_visible_page: {lev}  ({lev/len(d)*100:.1f}%)')

Total rows: 30000 unique clients: 32

Declining pages: 16262 / 30000 (54.2%)
...of which also have 100+ impressions: 13152 (43.8% of all pages)

Median avg_position: 10.8 (50% of pages sit around page 1-2 or worse)
Pages with 500+ impressions, pos 1-20, and low CTR (<0.5%): 9759 (32.5%)

Content age (days):
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
  Pages older than 6 months (>=180 days): 17986 / 30000

Trend direction breakdown:
down      16262
stable     5962
up         4388
new        2236
flat       1152

Baseline reason code hit rates (how many pages each rule flags):
  stale_visible_page: 17  (0.1%)
  declining_with_demand: 13152  (43.8%)
  page_one_decay_risk: 7076  (23.6%)
  low_ctr_visible_page: 9759  (32.5%)
  low_engagement_visible_page: 7113  (23.7%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This project builds a **decision-support ranking**. Here is exactly what that does and doesn’t mean:

**I can claim:**
- That certain observed signals (trend direction, impressions, position, CTR, content age, engagement) correlate with pages that a reviewer would want to look at first.
- That a learned ranking on these signals beats a fixed hand-coded rule on the same data — measured by precision@K, average precision, and similar metrics.
- Directional statements like “pages with these characteristics tend to appear higher in the review queue.”
- That the ranked queue helps a human reviewer prioritise their limited time, compared to having no queue at all.

**I cannot claim:**
- That editing a page will cause it to recover in search rankings. The data is observational, not experimental. Correlation is not causation.
- That I have identified Google’s ranking algorithm factors. This dataset contains no such information.
- That a refresh guarantees any specific outcome (traffic recovery, position improvement, CTR increase).
- Any statements about client identities, real URLs, raw queries, or keywords — the data is pseudonymized and I will not attempt to reverse that.

I follow the public-safe output rules from the lane guide (section 14): aggregated metrics, pseudonymized IDs, observed/directional language only. No client names, domains, raw queries, or causal claims will appear in any deliverable.

One more caveat worth naming now: the 54.2% “declining” figure and everything downstream of it uses `trend_direction == “down”`, which is a current-window proxy label computed from the same window — not a future outcome. The lane guide itself flags this as a starting point, not a final target. A stronger version of this project will define decline as a **future** outcome once I move to the full warehouse release: features from a prior 90-day window predicting decline or recovery over the next 30 days. That shift is the natural progression from Week 1 framing to the capstone data contract. Calling this out now so I don’t mistake a provisional label for the real thing.

In [4]:
# (Section 4 is a text section)

## Self-check

Before you submit, confirm each line honestly:

- [X] **Lane named** — Lane 2 (Refresh / Content Opportunity Scoring), with a concrete reason based on the data (54% declining, 44% declining with demand, validated baseline-to-RF gap).
- [X] **Decision + action named** — which page to review first; content reviewer audits the top-K queue; cost of false positive (wasted reviewer time) vs. false negative (missed decliner).
- [X] **2-3 real numbers from THIS data** — section 3 loads the actual CSV and computes real counts (16,262 declining, 13,152 declining with demand, 9,759 low-CTR visible pages, 7,076 page-one decay risks). These are not copied from the guide.
- [X] **Explain why this isn’t just “train a model”** — section 2 frames it as decision-support for a human reviewer with limited capacity, not automation. Section 4 rules out causal claims explicitly.
- [X] **Careful language used** — section 4 draws a clean line between observed/directional claims (allowed) and causal proofs (not allowed). No claims about Google’s algorithm, no “guaranteed recovery,” no client-identifying information.
- [X] **Notebook runs top to bottom** — all cells execute without errors.
- [X] **No client names, URLs, or private queries** — data is pseudonymized; no such fields appear in the notebook.

Committed to `work/notebooks/w01_research_question.ipynb`. Done.